# Lineup Consistency

This notebook measures how stable each team's batting order is across a season. The original scratch version counted exact lineup-slot matches between every pair of games; this version keeps that idea and adds a few complementary views:

- exact slot similarity: same player in the same batting-order slot
- roster similarity: same starters, regardless of batting-order slot
- order-aware similarity: same starters get partial credit when they move spots
- day-to-day consistency versus season-wide consistency
- repeated lineup templates
- lineup-slot entropy and player-slot entropy
- optional comparison to team wins/runs from the `mlb-predictions` CSVs

In [ ]:
from __future__ import annotations

import os
from functools import lru_cache
from io import StringIO
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'batting-order-matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(Path('/tmp') / 'batting-order-cache'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

In [ ]:
YEAR = '2025'
OUTDIR = ''
PREDICTIONS_BASE_URL = 'https://raw.githubusercontent.com/fantasy-toolz/mlb-predictions/refs/heads/main/data'
FETCH_PREDICTIONS = False

ALL_TEAMS = [
    'LAA', 'HOU', 'ATH', 'TOR', 'ATL', 'MIL', 'STL', 'CHC', 'AZ', 'LAD',
    'SF', 'CLE', 'SEA', 'MIA', 'NYM', 'WSH', 'BAL', 'SD', 'PHI', 'PIT',
    'TEX', 'TB', 'BOS', 'CIN', 'COL', 'KC', 'DET', 'MIN', 'CWS', 'NYY',
]

TEAMS = ALL_TEAMS
LINEUP_COLS = [f'lineup{i}' for i in range(1, 10)]


def find_data_root() -> Path:
    for candidate in [Path('../../data'), Path('../data'), Path('data')]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the repository data directory from this working directory.')


DATA_ROOT = find_data_root()
DATA_ROOT

In [ ]:
def lineup_path(team: str, year: str = YEAR, outdir: str = OUTDIR) -> Path:
    return DATA_ROOT / outdir / year / f'{team}.csv'


def read_team_lineups(team: str, year: str = YEAR, outdir: str = OUTDIR) -> pd.DataFrame:
    df = pd.read_csv(lineup_path(team, year, outdir))
    unnamed_cols = [col for col in df.columns if str(col).startswith('Unnamed')]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    df['team'] = team
    df['team_game'] = np.arange(len(df))
    df['date'] = df['date'].astype(str)
    df['date_base'] = df['date'].str.replace(r'[ab]$', '', regex=True)
    df['date_suffix'] = df['date'].str.extract(r'([ab])$', expand=False).fillna('')

    for col in LINEUP_COLS:
        df[col] = df[col].astype('string').str.strip()

    return df


def read_all_team_lineups(teams: list[str] = TEAMS) -> dict[str, pd.DataFrame]:
    return {team: read_team_lineups(team) for team in teams}


def lineup_long(df: pd.DataFrame) -> pd.DataFrame:
    long = df.melt(
        id_vars=['team', 'team_game', 'date', 'date_base', 'date_suffix'],
        value_vars=LINEUP_COLS,
        var_name='lineup_slot',
        value_name='player',
    )
    long = long.dropna(subset=['player'])
    long = long[long['player'] != '']
    long['lineup_spot'] = long['lineup_slot'].str.extract(r'(\d+)').astype(int)
    return long.drop(columns=['lineup_slot'])


def lineup_values(df: pd.DataFrame) -> np.ndarray:
    return df[LINEUP_COLS].astype(str).to_numpy()

In [ ]:
def entropy(values: pd.Series) -> float:
    counts = values.dropna().value_counts()
    if counts.empty:
        return 0.0
    probabilities = counts / counts.sum()
    return float(-(probabilities * np.log2(probabilities)).sum())


def upper_triangle_values(matrix: np.ndarray) -> np.ndarray:
    if matrix.shape[0] < 2:
        return np.array([])
    idx = np.triu_indices(matrix.shape[0], k=1)
    return matrix[idx]


def adjacent_values(matrix: np.ndarray) -> np.ndarray:
    if matrix.shape[0] < 2:
        return np.array([])
    return np.diag(matrix, k=1)


def exact_slot_similarity_matrix(df: pd.DataFrame) -> np.ndarray:
    lineups = lineup_values(df)
    return (lineups[:, None, :] == lineups[None, :, :]).mean(axis=2)


def roster_similarity_matrix(df: pd.DataFrame) -> np.ndarray:
    lineups = lineup_values(df)
    roster_sets = [set(row) for row in lineups]
    n_games = len(roster_sets)
    matrix = np.eye(n_games)

    for i in range(n_games):
        for j in range(i + 1, n_games):
            score = len(roster_sets[i] & roster_sets[j]) / len(LINEUP_COLS)
            matrix[i, j] = score
            matrix[j, i] = score

    return matrix


def order_aware_similarity_matrix(df: pd.DataFrame) -> np.ndarray:
    lineups = lineup_values(df)
    position_maps = [
        {player: slot for slot, player in enumerate(row, start=1)}
        for row in lineups
    ]
    n_games = len(position_maps)
    matrix = np.eye(n_games)

    for i in range(n_games):
        for j in range(i + 1, n_games):
            shared_players = set(position_maps[i]) & set(position_maps[j])
            score = 0.0
            for player in shared_players:
                slot_distance = abs(position_maps[i][player] - position_maps[j][player])
                score += 1 - (slot_distance / (len(LINEUP_COLS) - 1))
            score /= len(LINEUP_COLS)
            matrix[i, j] = score
            matrix[j, i] = score

    return matrix


def similarity_matrices(df: pd.DataFrame) -> dict[str, np.ndarray]:
    return {
        'exact_slot': exact_slot_similarity_matrix(df),
        'roster': roster_similarity_matrix(df),
        'order_aware': order_aware_similarity_matrix(df),
    }

In [ ]:
def lineup_templates(df: pd.DataFrame) -> pd.DataFrame:
    template_series = df[LINEUP_COLS].astype(str).apply(tuple, axis=1)
    counts = template_series.value_counts().rename_axis('lineup_template').rename('games').reset_index()
    counts['share'] = counts['games'] / len(df)
    counts['lineup'] = counts['lineup_template'].apply(lambda lineup: ' | '.join(lineup))
    return counts[['lineup', 'games', 'share', 'lineup_template']]


def slot_entropy_table(df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for slot, col in enumerate(LINEUP_COLS, start=1):
        counts = df[col].value_counts()
        modal_player = counts.index[0] if len(counts) else np.nan
        modal_games = int(counts.iloc[0]) if len(counts) else 0
        rows.append({
            'lineup_spot': slot,
            'unique_players': int(df[col].nunique()),
            'entropy': entropy(df[col]),
            'modal_player': modal_player,
            'modal_games': modal_games,
            'modal_share': modal_games / len(df) if len(df) else np.nan,
        })
    return pd.DataFrame(rows)


def player_slot_table(df: pd.DataFrame, min_starts: int = 5) -> pd.DataFrame:
    long = lineup_long(df)
    rows: list[dict[str, object]] = []

    for player, group in long.groupby('player'):
        starts = len(group)
        if starts < min_starts:
            continue
        slot_counts = group['lineup_spot'].value_counts().sort_index()
        modal_slot = int(slot_counts.idxmax())
        modal_games = int(slot_counts.max())
        rows.append({
            'player': player,
            'starts': starts,
            'unique_slots': int(group['lineup_spot'].nunique()),
            'slot_entropy': entropy(group['lineup_spot']),
            'modal_slot': modal_slot,
            'modal_slot_games': modal_games,
            'modal_slot_share': modal_games / starts,
            'avg_slot': float(group['lineup_spot'].mean()),
            'slot_std': float(group['lineup_spot'].std()) if starts > 1 else 0.0,
        })

    return pd.DataFrame(rows).sort_values(['starts', 'modal_slot_share'], ascending=[False, False]).reset_index(drop=True)

In [ ]:
def team_consistency_summary(team: str, df: pd.DataFrame) -> dict[str, object]:
    matrices = similarity_matrices(df)
    template_counts = lineup_templates(df)
    slot_entropy = slot_entropy_table(df)
    player_slots = player_slot_table(df)

    summary: dict[str, object] = {
        'team': team,
        'games': len(df),
        'unique_lineups': len(template_counts),
        'unique_lineup_pct': len(template_counts) / len(df) if len(df) else np.nan,
        'most_common_lineup_games': int(template_counts['games'].iloc[0]) if len(template_counts) else 0,
        'most_common_lineup_pct': float(template_counts['share'].iloc[0]) if len(template_counts) else np.nan,
        'top_5_lineup_pct': float(template_counts['share'].head(5).sum()) if len(template_counts) else np.nan,
        'mean_slot_entropy': float(slot_entropy['entropy'].mean()) if len(slot_entropy) else np.nan,
        'max_slot_entropy': float(slot_entropy['entropy'].max()) if len(slot_entropy) else np.nan,
        'mean_player_slot_entropy': float(player_slots['slot_entropy'].mean()) if len(player_slots) else np.nan,
        'stable_player_count': int((player_slots['modal_slot_share'] >= 0.75).sum()) if len(player_slots) else 0,
    }

    for name, matrix in matrices.items():
        pair_values = upper_triangle_values(matrix)
        adjacent = adjacent_values(matrix)
        summary[f'mean_{name}_similarity'] = float(pair_values.mean()) if len(pair_values) else np.nan
        summary[f'median_{name}_similarity'] = float(np.median(pair_values)) if len(pair_values) else np.nan
        summary[f'day_to_day_{name}_similarity'] = float(adjacent.mean()) if len(adjacent) else np.nan
        summary[f'min_day_to_day_{name}_similarity'] = float(adjacent.min()) if len(adjacent) else np.nan

    return summary


def build_team_summary(team_games: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = [team_consistency_summary(team, df) for team, df in team_games.items()]
    return pd.DataFrame(rows).sort_values('day_to_day_roster_similarity', ascending=False).reset_index(drop=True)

In [ ]:
def prediction_url(team: str, year: str = YEAR) -> str:
    return f'{PREDICTIONS_BASE_URL}/{year}/teams/{team}.csv'


@lru_cache(maxsize=64)
def read_team_predictions(team: str, year: str = YEAR) -> pd.DataFrame:
    response = requests.get(prediction_url(team, year), timeout=30)
    response.raise_for_status()
    predictions = pd.read_csv(StringIO(response.text))
    predictions['date'] = predictions['date'].astype(str)
    return predictions


def team_prediction_summary(team: str, year: str = YEAR) -> dict[str, object]:
    predictions = read_team_predictions(team, year)
    summary: dict[str, object] = {'team': team}
    if 'rundifferential' in predictions.columns:
        summary['wins'] = int((predictions['rundifferential'] > 0).sum())
        summary['losses'] = int((predictions['rundifferential'] < 0).sum())
        summary['mean_run_differential'] = float(predictions['rundifferential'].mean())
    if 'teamruns' in predictions.columns:
        summary['runs_per_game'] = float(predictions['teamruns'].mean())
    if 'teamruns6' in predictions.columns:
        summary['six_inning_runs_per_game'] = float(predictions['teamruns6'].mean())
    return summary


def add_prediction_summary(team_summary: pd.DataFrame, teams: list[str] = TEAMS) -> pd.DataFrame:
    prediction_summary = pd.DataFrame([team_prediction_summary(team) for team in teams])
    return team_summary.merge(prediction_summary, on='team', how='left')

In [ ]:
def plot_similarity_heatmap(df: pd.DataFrame, metric: str = 'exact_slot', title: str | None = None) -> None:
    matrices = similarity_matrices(df)
    matrix = matrices[metric]
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap='viridis')
    ax.set_xlabel('Team game')
    ax.set_ylabel('Team game')
    ax.set_title(title or f'{metric} lineup similarity')
    fig.colorbar(image, ax=ax, label='Similarity')
    plt.tight_layout()


def plot_day_to_day_similarity(df: pd.DataFrame, title: str | None = None) -> None:
    matrices = similarity_matrices(df)
    x = np.arange(1, len(df))
    fig, ax = plt.subplots(figsize=(12, 4))
    for metric, matrix in matrices.items():
        ax.plot(x, adjacent_values(matrix), label=metric.replace('_', ' '), linewidth=1.8)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Team game')
    ax.set_ylabel('Similarity to previous game')
    ax.set_title(title or 'Day-to-day lineup similarity')
    ax.legend()
    plt.tight_layout()


def plot_slot_entropy(df: pd.DataFrame, title: str | None = None) -> None:
    slots = slot_entropy_table(df)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(slots['lineup_spot'], slots['entropy'], color='tab:blue')
    ax.set_xticks(range(1, 10))
    ax.set_xlabel('Lineup spot')
    ax.set_ylabel('Entropy')
    ax.set_title(title or 'Lineup spot entropy')
    plt.tight_layout()


def plot_player_slot_heatmap(df: pd.DataFrame, min_starts: int = 10, title: str | None = None) -> None:
    long = lineup_long(df)
    counts = (
        long.groupby(['player', 'lineup_spot'])
        .size()
        .rename('starts')
        .reset_index()
    )
    player_totals = counts.groupby('player')['starts'].sum().sort_values(ascending=False)
    players = player_totals[player_totals >= min_starts].index.tolist()
    pivot = (
        counts[counts['player'].isin(players)]
        .pivot(index='player', columns='lineup_spot', values='starts')
        .fillna(0)
        .reindex(players)
    )

    fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(pivot))))
    image = ax.imshow(pivot.to_numpy(), aspect='auto', cmap='magma')
    ax.set_xticks(np.arange(9), labels=np.arange(1, 10))
    ax.set_yticks(np.arange(len(pivot)), labels=pivot.index)
    ax.set_xlabel('Lineup spot')
    ax.set_title(title or 'Player starts by lineup spot')
    fig.colorbar(image, ax=ax, label='Starts')
    plt.tight_layout()


def plot_consistency_scatter(summary: pd.DataFrame, x: str, y: str, title: str | None = None) -> None:
    plot_df = summary.dropna(subset=[x, y])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(plot_df[x], plot_df[y], color='black', s=28, alpha=0.8)
    for _, row in plot_df.iterrows():
        ax.annotate(row['team'], (row[x], row[y]), fontsize=8, alpha=0.75)
    ax.set_xlabel(x.replace('_', ' '))
    ax.set_ylabel(y.replace('_', ' '))
    ax.set_title(title or f'{y} vs {x}')
    plt.tight_layout()

In [ ]:
team_games = read_all_team_lineups(TEAMS)
team_summary = build_team_summary(team_games)

if FETCH_PREDICTIONS:
    team_summary = add_prediction_summary(team_summary, TEAMS)

team_summary.head(30)

In [ ]:
summary_columns = [
    'team',
    'games',
    'day_to_day_roster_similarity',
    'day_to_day_exact_slot_similarity',
    'day_to_day_order_aware_similarity',
    'mean_roster_similarity',
    'mean_exact_slot_similarity',
    'unique_lineup_pct',
    'most_common_lineup_pct',
    'top_5_lineup_pct',
    'mean_slot_entropy',
    'stable_player_count',
]
team_summary[summary_columns].sort_values('day_to_day_roster_similarity', ascending=False)

In [ ]:
TEAM = 'MIN'
team_df = team_games[TEAM]

team_consistency_summary(TEAM, team_df)

In [ ]:
lineup_templates(team_df).head(10)

In [ ]:
slot_entropy_table(team_df)

In [ ]:
player_slot_table(team_df).head(20)

In [ ]:
plot_similarity_heatmap(team_df, metric='exact_slot', title=f'{TEAM}: exact slot similarity')

In [ ]:
plot_day_to_day_similarity(team_df, title=f'{TEAM}: day-to-day lineup similarity')

In [ ]:
plot_slot_entropy(team_df, title=f'{TEAM}: lineup spot entropy')

In [ ]:
plot_player_slot_heatmap(team_df, min_starts=10, title=f'{TEAM}: player/slot starts')

In [ ]:
# After setting FETCH_PREDICTIONS = True and rerunning the summary cell, these comparisons become available.
if {'wins', 'runs_per_game'}.issubset(team_summary.columns):
    plot_consistency_scatter(team_summary, 'wins', 'day_to_day_roster_similarity')
    plot_consistency_scatter(team_summary, 'runs_per_game', 'day_to_day_roster_similarity')

## Reading The Metrics

- `day_to_day_roster_similarity` answers: how many of yesterday's starters are usually back in today's lineup?
- `day_to_day_exact_slot_similarity` answers: how often are players back in the same batting-order slot the next game?
- `day_to_day_order_aware_similarity` gives partial credit when the same players are used but shuffled.
- `mean_*_similarity` compares all game pairs, so it measures season-wide consistency rather than local day-to-day consistency.
- `unique_lineup_pct` near 1.0 means almost every game used a distinct lineup template.
- `top_5_lineup_pct` measures how much of the season is covered by the team's five most common exact lineups.
- high slot entropy means that lineup spot churned through many players or lacked a dominant regular.
- high player slot entropy means a player moved around the batting order rather than owning one role.